# Ollama

In [1]:
!pip cache purge --quiet

In [2]:
!pip install ollama==0.6.0 \
             numpy==2.3.4 \
             pandas==2.3.3 \
             --no-deps --quiet

In [3]:
import json
import logging
import numpy as np
import ollama
import os
import pandas as pd

from urllib.parse import quote

In [4]:
logging.getLogger("httpx").setLevel(logging.ERROR)

In [5]:
EMBEDDING_MODEL = "all-minilm"
LLM_MODEL = "llama3"

In [6]:
ollama.pull(EMBEDDING_MODEL)

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [7]:
ollama.pull(LLM_MODEL)

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [8]:
company_info = {
    "AAPL": "Apple Inc. is a technology company known for the iPhone, iPad and Mac. It also offers services like iCloud and Apple Music.",
    "AMZN": "Amazon.com is an e-commerce giant with businesses in cloud computing (AWS), logistics and digital streaming.",
    "GOOG": "Google, a subsidiary of Alphabet Inc., is a technology company specializing in internet services, online advertising, search and AI.",
    "MSFT": "Microsoft develops software such as Windows and Office and is a leader in cloud computing with Azure."
}

In [9]:
df_data = []

for sym, text in company_info.items():
    response = ollama.embeddings(
        model = EMBEDDING_MODEL,
        prompt = text
    )
    embedding_array = np.array(response["embedding"], dtype = np.float32)
    df_data.append({"content": text, "vector": embedding_array, "metadata": json.dumps({"symbol": sym})})

df = pd.DataFrame(df_data)

# Call the embedding model once with a dummy string
response = ollama.embeddings(
    model = EMBEDDING_MODEL,
    prompt = "test"
)

# Extract dimension
dimensions = len(response["embedding"])

In [10]:
username = "admin"
password = os.environ.get("SINGLESTOREDB_PASSWORD")
host = os.environ.get("SINGLESTOREDB_HOST")
port = 3306
database = "ollama_db"

if not password:
    raise ValueError("Environment variable SINGLESTOREDB_PASSWORD is not set or is empty.")

if not host:
    raise ValueError("Environment variable SINGLESTOREDB_HOST is not set or is empty.")

problematic_chars = ["#", "@", "/", "?", "%"]
found = [c for c in problematic_chars if c in password]
if found:
    print(f"WARNING: Password contains character(s) {found} which may cause connection issues.")

my_connection_url = f"singlestoredb://{username}:{quote(password, safe='')}@{host}:{port}/{database}"

In [11]:
from sqlalchemy import *

db_connection = create_engine(my_connection_url)

In [12]:
try:
    with db_connection.begin() as conn:
        conn.execute(text("DROP TABLE IF EXISTS company_knowledge;"))
except Exception as e:
    print(f"Error dropping table: {e}")
    raise

In [13]:
query = text("""
CREATE TABLE IF NOT EXISTS company_knowledge (
    id BIGINT AUTO_INCREMENT NOT NULL PRIMARY KEY,
    content LONGTEXT,
    vector VECTOR(:dimensions) NOT NULL,
    metadata JSON,
    VECTOR INDEX (vector) INDEX_OPTIONS '{"metric_type": "DOT_PRODUCT"}'
);
""")

with db_connection.begin() as conn:
    conn.execute(query, {"dimensions": dimensions})

In [14]:
df.to_sql(
    "company_knowledge",
    con = db_connection,
    if_exists = "append",
    index = False,
    chunksize = 1000
)

4

In [15]:
prompt = "What are the most popular consumer devices and services that Apple Inc. sells?"

response = ollama.embeddings(
    model = EMBEDDING_MODEL,
    prompt = prompt
)
embedding_array = np.array(response["embedding"], dtype = np.float32)

query = text("""
SELECT content
FROM company_knowledge
ORDER BY vector <*> :embedding_array DESC
LIMIT 1;
""")

with db_connection.connect() as conn:
    row = conn.execute(query, {"embedding_array": embedding_array}).fetchone()

data = row[0]
print(data)

Apple Inc. is a technology company known for the iPhone, iPad and Mac. It also offers services like iCloud and Apple Music.


In [16]:
output = ollama.generate(
    model = LLM_MODEL,
    prompt = f"Using this data: {data}. Respond to this prompt: {prompt}",
    options = {
        "temperature": 0
    }
)

print(output["response"])

Based on the provided data, the most popular consumer devices sold by Apple Inc. are:

1. iPhone
2. iPad
3. Mac (computers)

As for services, Apple Inc. offers:

1. iCloud (cloud storage and backup)
2. Apple Music (music streaming)

These products and services are some of the most well-known and widely used among consumers, making them the most popular offerings from Apple Inc.
